In [39]:

import websocket
import json
import csv
import datetime
import os
import pandas as pd
import threading
import time



In [40]:

WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
CSV_FILE_NAME = "data.csv"

In [41]:

def tao_file_csv():
    header = ["thoi_gian", "gia", "gia_mua", "gia_ban", "khoi_luong_24h","high24h", "low24h","instId","best_purchase_price", "best_sale_price"]
    
    if not os.path.exists(CSV_FILE_NAME) or os.path.getsize(CSV_FILE_NAME) == 0:
        with open(CSV_FILE_NAME, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV: {CSV_FILE_NAME}")

tao_file_csv()

In [42]:

def on_open(ws):
    print(f" Đã kết nối thành công")
    
    
    for inst_id in INSTRUMENT_IDS:
        subscribe_message = {
            "op": "subscribe",
            "args": [
                {    
                    "instType": "USDT-FUTURES",
                    "channel": "ticker",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(subscribe_message))
        print(f"Đang theo dõi {inst_id}")
    print(f"Đang theo dõi các cặp: {', '.join(INSTRUMENT_IDS)}")

all_data= []# nơi lưu dữ liệu tạm thời

def on_message(ws, message_str):
    global all_data
    data = json.loads(message_str)
    all_data.append(data)
    
    if "data" in data and data["data"]:
        ticker = data["data"][0]
        
        
        thoi_gian = datetime.datetime.now().isoformat()
        gia = ticker.get('lastPr')
        gia_mua = ticker.get('bidPr')
        gia_ban = ticker.get('askPr')
        khoi_luong = ticker.get('volumeUsd24h')
        high_int_24h= ticker.get('high24h')
        low_int_24h = ticker.get('low24h')
        instId = ticker.get('instId') 
        best_purchase_price= ticker.get('bidSz')
        best_sale_price = ticker.get('askSz')
        
        
        # Hiển thị
        print(f" {instId} | Giá: {gia}") 
        
        # Lưu vào CSV
        with open(CSV_FILE_NAME, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([thoi_gian, gia, gia_mua, gia_ban, khoi_luong, high_int_24h, low_int_24h, instId, best_purchase_price, best_sale_price])

def on_error(ws, error):
    print(f" Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f" Kết nối đã đóng")

In [ ]:
a=60  #1 phút
b=a*60  #1h
c=b*24  #1 ngày
def run_ws():
    ws.run_forever(ping_interval=30, ping_timeout=10)
ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open,
                          on_message=on_message,
                          on_error=on_error,
                          on_close=on_close)

print(f"Bắt đầu kết nối đến Bitget...")
print(f"Dữ liệu sẽ được lưu vào: {CSV_FILE_NAME}")
print("Nhấn Ctrl+C để dừng")

ws_thread = threading.Thread(target=run_ws)
ws_thread.daemon = True
ws_thread.start()

run_duration = 10

try:
    time.sleep(run_duration)
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")

# Đóng kết nối sau thời gian quy định
ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")

# nhìn time ở trên tự tính hoặc để mặc định 
# add time ở phần Run_duration
# rồi chạy cell dataframe
# muốn xóa  phần add vào csv thì xóa phần CSV_FILE_NAME = "data.csv" ở trên


Bắt đầu kết nối đến Bitget...
Dữ liệu sẽ được lưu vào: data.csv
Nhấn Ctrl+C để dừng
 Đã kết nối thành công
Đang theo dõi SOLUSDT
Đang theo dõi BTCUSDT
Đang theo dõi ETHUSDT
Đang theo dõi các cặp: SOLUSDT, BTCUSDT, ETHUSDT
 ETHUSDT | Giá: 2632
 BTCUSDT | Giá: 108673
 SOLUSDT | Giá: 175.024
 ETHUSDT | Giá: 2632
 SOLUSDT | Giá: 175.024
 BTCUSDT | Giá: 108673
 BTCUSDT | Giá: 108673
 ETHUSDT | Giá: 2632.01
 SOLUSDT | Giá: 175.024
 ETHUSDT | Giá: 2632.01
 BTCUSDT | Giá: 108673
 SOLUSDT | Giá: 175.024
 ETHUSDT | Giá: 2632.01
 BTCUSDT | Giá: 108673
 SOLUSDT | Giá: 175.024
 BTCUSDT | Giá: 108673
 ETHUSDT | Giá: 2632.01
 SOLUSDT | Giá: 175.024
 ETHUSDT | Giá: 2632.01
 BTCUSDT | Giá: 108673
 SOLUSDT | Giá: 175.024
 BTCUSDT | Giá: 108673.1
 ETHUSDT | Giá: 2632
 SOLUSDT | Giá: 175.024
 BTCUSDT | Giá: 108673.1
 ETHUSDT | Giá: 2632
 SOLUSDT | Giá: 175.024
 ETHUSDT | Giá: 2632
 BTCUSDT | Giá: 108673.1
 SOLUSDT | Giá: 175.024
 ETHUSDT | Giá: 2632
 BTCUSDT | Giá: 108673
 SOLUSDT | Giá: 175.024
 BTCUSDT 

In [44]:

ws.close()


data_for_df = []
for msg in all_data:
    if "data" in msg and msg["data"]:
        ticker = msg["data"][0]
        
        row = {
            "thoi_gian": datetime.datetime.now().isoformat(), 
            "gia": ticker.get('lastPr'),
            "gia_mua": ticker.get('bidPr'),
            "gia_ban": ticker.get('askPr'),
            "khoi_luong_24h": ticker.get('volumeUsd24h'),
            "high24h": ticker.get('high24h'),
            "low24h": ticker.get('low24h'),
            "instId": ticker.get('instId'),
            "best_purchase_price": ticker.get('bidSz'),
            "best_sale_price": ticker.get('askSz')
        }
        data_for_df.append(row)


df = pd.DataFrame(data_for_df)

